# RIME-X Multi-Model Pipeline
### Processes all 4 GCM models automatically: CNRM-ESM2-1, IPSL-CM6A-LR, MPI-ESM1-2-LR, UKESM1-0-LL

**What this notebook does:**
1. Loads the no-skip nearest-neighbour masks (built previously)
2. Loops through all 4 models and all their scenario files
3. Computes regional averages for every region × every timestep
4. Saves CSVs ready for `rime-pre-quantilemap`


## Cell 1 — Imports

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import os, gc, time
from tqdm.notebook import tqdm
from datetime import datetime

print("Imports done ✅")
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


## Cell 2 — Configuration

In [ ]:
# ── ROOT where all GCM_models folders live ───────────────────────────
ROOT = '/media/wcs/Disk3/imran/GCM_models'

# ── MODEL CONFIG: name → folder name on disk ──────────────────────────
# Update folder names if they differ on your machine
MODELS = {
    'CNRM-ESM2-1':   'CNRM-ESM2-1',
    'IPSL-CM6A-LR':  'IPSL-CM6A-LR',
    'MPI-ESM1-2-LR': 'MPI-ESM1-2-LR',
    'UKESM1-0-LL':   'UKESM1-0-LL',
}

# ── INDICATOR ─────────────────────────────────────────────────────────
INDICATOR = 'tas'

# ── SCENARIOS to include ──────────────────────────────────────────────
SCENARIOS_I_WANT = ['historical','ssp126','ssp245','ssp370','ssp460','ssp585']

# ── MASKS (built in previous notebook) ───────────────────────────────
MASKS_PATH = f'{ROOT}/masks/subregion_masks_noskip.nc'

# ── OUTPUT ROOT ───────────────────────────────────────────────────────
# Each model gets its own subfolder so outputs stay organized
OUTPUT_ROOT = f'{ROOT}/all_models_regional_output'

# ── VERIFY all model folders exist ───────────────────────────────────
print("Model folder check:")
for model, folder in MODELS.items():
    sim_dir = f'{ROOT}/{folder}/RegridedMonthly'
    exists  = os.path.isdir(sim_dir)
    files   = len([f for f in os.listdir(sim_dir) if f.endswith('.nc')]) if exists else 0
    status  = f"✅ found ({files} .nc files)" if exists else "❌ NOT FOUND"
    print(f"  {model:20s} → {sim_dir}")
    print(f"  {'':20s}   {status}")

print(f"\nMasks path: {MASKS_PATH}")
print(f"Masks exist: {os.path.exists(MASKS_PATH)} ✅" if os.path.exists(MASKS_PATH) 
      else f"Masks exist: ❌ NOT FOUND — run cell6_noskip.ipynb first")
print(f"\nOutput root: {OUTPUT_ROOT}")


## Cell 3 — Load Masks (run once)

In [ ]:
# Load ALL masks into memory as a single numpy matrix
# This is done ONCE and reused for all 4 models
# Memory: ~152 MB — stays constant regardless of how many models you run

print("Loading mask dataset...")
masks_ds     = xr.open_dataset(MASKS_PATH)
region_names = list(masks_ds.data_vars)
lat_vals     = masks_ds.lat.values
lon_vals     = masks_ds.lon.values

n_regions = len(region_names)
n_lat     = len(lat_vals)
n_lon     = len(lon_vals)
n_cells   = n_lat * n_lon

print(f"  Regions  : {n_regions}")
print(f"  Grid     : {n_lat} lat × {n_lon} lon = {n_cells} cells")

# cos(lat) weights
cos_lat_2d   = np.cos(np.deg2rad(lat_vals))[:, np.newaxis] * np.ones((1, n_lon))
cos_lat_2d   = cos_lat_2d.astype(np.float32)
cos_lat_flat = cos_lat_2d.ravel()                        # (n_cells,)

# Build mask matrix (n_regions, n_cells)
print("\nBuilding mask matrix (float32)...")
masks_matrix = np.zeros((n_regions, n_cells), dtype=np.float32)

for i, region in enumerate(tqdm(region_names, desc="Loading mask vars")):
    mask_2d          = masks_ds[region].values.astype(np.float32)
    weighted         = mask_2d * cos_lat_2d
    weighted[weighted == 0] = np.nan
    masks_matrix[i]  = weighted.ravel()

masks_ds.close()

# Denominators per region
denominators = np.nansum(masks_matrix, axis=1)           # (n_regions,)

# Clean matrix for multiplication (NaN → 0)
masks_clean = np.where(np.isnan(masks_matrix), 0.0, masks_matrix)
del masks_matrix
gc.collect()

print(f"\nMask matrix ready ✅")
print(f"  Shape    : {masks_clean.shape}")
print(f"  Memory   : {masks_clean.nbytes / 1e6:.1f} MB")
print(f"  Valid regions (denom > 0): {(denominators > 0).sum()} / {n_regions}")


## Cell 4 — Discover All Files Across All Models

In [ ]:
# Build a complete inventory of all files to process

all_tasks = []   # list of (model_name, sim_dir, target_dir, filename)

print("File inventory:")
print("=" * 65)

for model_name, folder in MODELS.items():
    sim_dir    = f'{ROOT}/{folder}/RegridedMonthly'
    target_dir = f'{OUTPUT_ROOT}/{model_name}'

    if not os.path.isdir(sim_dir):
        print(f"  {model_name:20s} ❌ RegridedMonthly not found — skipping")
        continue

    files = [
        f for f in os.listdir(sim_dir)
        if f.endswith('.nc')
        and len(f.split('_')) > 3
        and f.split('_')[3] in SCENARIOS_I_WANT
    ]

    print(f"\n  {model_name} ({len(files)} files):")
    for f in files:
        parts = f.split('_')
        print(f"    scenario={parts[3]:12s}  ensemble={parts[4]:15s}  → {f}")
        all_tasks.append((model_name, sim_dir, target_dir, f))

print(f"\n{'=' * 65}")
print(f"Total files to process: {len(all_tasks)}")


## Cell 5 — Define Processing Function

In [ ]:
def process_one_file(
    simulation_directory,
    simulation_file,
    target_directory,
    masks_clean,
    denominators,
    region_names,
    cos_lat_flat,
    indicator,
    need_global_mean=False,
    verbose=True,
):
    """
    Computes area-weighted regional averages for one NetCDF file.
    Uses pure NumPy matrix multiplication — minimal RAM usage (~200MB).
    Returns: (n_written, n_skipped, elapsed_seconds)
    """
    simulation_path = f"{simulation_directory}/{simulation_file}"

    # Parse metadata from filename
    parts     = simulation_file.split('_')
    MODEL     = parts[2]
    scenario  = parts[3]
    ensemble  = parts[4]
    SCENARIO  = f"{scenario}-{ensemble}"
    INDICATOR = parts[0]

    t0 = time.time()

    # ── Load simulation ───────────────────────────────────────────────
    with xr.open_dataset(simulation_path) as ds:
        ds['lon'] = xr.where(ds['lon'] > 180, ds['lon'] - 360, ds['lon'])
        ds        = ds.sortby('lon')
        sim_np    = ds[indicator].values.astype(np.float32)   # (time, lat, lon)
        time_vals = ds['time'].values

    sim_flat = sim_np.reshape(sim_np.shape[0], -1)            # (time, n_cells)
    del sim_np
    gc.collect()

    # ── Global mean ───────────────────────────────────────────────────
    if need_global_mean:
        denom_g   = cos_lat_flat.sum()
        global_ts = (sim_flat * cos_lat_flat[np.newaxis, :]).sum(axis=1) / denom_g
        df_g      = pd.DataFrame({'time': time_vals, 'tas': global_ts})
        gm_dir    = f"{target_directory}/cmip-6_global_mean/{indicator}"
        os.makedirs(gm_dir, exist_ok=True)
        df_g.to_csv(
            f"{gm_dir}/globalmean_{INDICATOR.lower()}_{MODEL.lower()}_{SCENARIO.lower()}.csv",
            index=False
        )
        del global_ts, df_g

    # ── Matrix multiply: all regions × all timesteps at once ──────────
    # (n_time, n_cells) @ (n_cells, n_regions) → (n_time, n_regions)
    all_avg = sim_flat @ masks_clean.T
    all_avg /= denominators[np.newaxis, :]

    del sim_flat
    gc.collect()

    # ── Write CSVs ────────────────────────────────────────────────────
    n_written  = 0
    n_skipped  = 0

    for i, region in enumerate(region_names):
        if denominators[i] == 0:
            n_skipped += 1
            continue

        region_clean = region.replace('m_', '')
        df = pd.DataFrame({'time': time_vals, region_clean: all_avg[:, i]})

        out_dir  = f"{target_directory}/isimip_regional_data/{region_clean}/latWeight"
        filename = f"{MODEL}_{SCENARIO}_{INDICATOR}_{region_clean}_latweight.csv".lower()
        os.makedirs(out_dir, exist_ok=True)
        df.to_csv(f"{out_dir}/{filename}", index=False)
        n_written += 1

    del all_avg
    gc.collect()

    elapsed = time.time() - t0
    if verbose:
        print(f"    ✅ written={n_written}  skipped={n_skipped}  "
              f"time={elapsed:.1f}s")

    return n_written, n_skipped, elapsed


print("Function defined ✅")


## Cell 6 — Run Pipeline for All 4 Models

In [ ]:
# ── RUN EVERYTHING ───────────────────────────────────────────────────
need_global_mean = (INDICATOR == 'tas')

total_files    = len(all_tasks)
done           = 0
total_written  = 0
total_skipped  = 0
failed_files   = []

print(f"Starting pipeline: {total_files} files across {len(MODELS)} models")
print(f"Indicator   : {INDICATOR}")
print(f"Global mean : {need_global_mean}")
print(f"Regions     : {n_regions}")
print("=" * 65)
pipeline_start = time.time()

for model_name, sim_dir, target_dir, filename in all_tasks:

    parts    = filename.split('_')
    scenario = parts[3]
    ensemble = parts[4]

    print(f"\n[{done+1}/{total_files}] {model_name} | {scenario} | {ensemble}")

    try:
        n_written, n_skipped, elapsed = process_one_file(
            simulation_directory = sim_dir,
            simulation_file      = filename,
            target_directory     = target_dir,
            masks_clean          = masks_clean,
            denominators         = denominators,
            region_names         = region_names,
            cos_lat_flat         = cos_lat_flat,
            indicator            = INDICATOR,
            need_global_mean     = need_global_mean,
            verbose              = True,
        )
        total_written += n_written
        total_skipped += n_skipped

    except Exception as e:
        print(f"    ❌ FAILED: {e}")
        failed_files.append((model_name, filename, str(e)))

    done += 1

# ── Summary ───────────────────────────────────────────────────────────
total_elapsed = time.time() - pipeline_start
print(f"\n{'=' * 65}")
print(f"PIPELINE COMPLETE ✅")
print(f"  Files processed : {done} / {total_files}")
print(f"  Regions written : {total_written}")
print(f"  Regions skipped : {total_skipped}")
print(f"  Total time      : {total_elapsed/60:.1f} minutes")
print(f"  Finished        : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

if failed_files:
    print(f"\n⚠️  Failed files ({len(failed_files)}):")
    for model, fname, err in failed_files:
        print(f"    {model} | {fname}")
        print(f"    Error: {err}")


## Cell 7 — Verify Outputs

In [ ]:
# Full inventory of what was produced
print("OUTPUT SUMMARY")
print("=" * 65)

for model_name in MODELS:
    target_dir = f'{OUTPUT_ROOT}/{model_name}'

    # Global mean CSVs
    gm_dir = f'{target_dir}/cmip-6_global_mean/{INDICATOR}'
    gm_files = os.listdir(gm_dir) if os.path.exists(gm_dir) else []

    # Regional CSVs
    reg_dir  = f'{target_dir}/isimip_regional_data'
    regions  = os.listdir(reg_dir) if os.path.exists(reg_dir) else []

    print(f"\n{model_name}:")
    print(f"  Global mean CSVs : {len(gm_files)}")
    for f in gm_files:
        df = pd.read_csv(f"{gm_dir}/{f}")
        print(f"    {f}")
        print(f"    → {len(df)} rows, {df['time'].iloc[0]} to {df['time'].iloc[-1]}")

    print(f"  Regional folders : {len(regions)} / {n_regions}")
    print(f"  Sample regions   : {regions[:3]}")

    # Preview one regional CSV
    if regions:
        r   = regions[0]
        lw  = f"{reg_dir}/{r}/latWeight"
        if os.path.exists(lw):
            csvs = os.listdir(lw)
            print(f"  Sample CSV       : {csvs[0]}")
            df   = pd.read_csv(f"{lw}/{csvs[0]}")
            print(f"  Preview:")
            print(df.head(3).to_string(index=False))


## Cell 8 — Check Readiness for RIME-X

In [ ]:
# Confirms your data is correctly structured for rime-pre-quantilemap

print("RIME-X READINESS CHECK")
print("=" * 65)

all_ready = True

for model_name in MODELS:
    target_dir = f'{OUTPUT_ROOT}/{model_name}'
    gm_dir     = f'{target_dir}/cmip-6_global_mean/{INDICATOR}'
    reg_dir    = f'{target_dir}/isimip_regional_data'

    gm_ok  = os.path.exists(gm_dir)  and len(os.listdir(gm_dir))  > 0
    reg_ok = os.path.exists(reg_dir) and len(os.listdir(reg_dir)) > 0

    status = "✅ READY" if (gm_ok and reg_ok) else "❌ MISSING DATA"
    if not (gm_ok and reg_ok):
        all_ready = False

    print(f"  {model_name:20s}  GMT={gm_ok}  Regional={reg_ok}  → {status}")

print()
if all_ready:
    print("✅ All models ready — you can now run rime-pre-quantilemap")
    print()
    print("Next command:")
    print("  rime-pre-wl         ← compute warming levels from GMT CSVs")
    print("  rime-pre-quantilemap ← build quantile maps from regional CSVs")
else:
    print("⚠️  Some models missing — check failed_files list above")
